Transformer解析  
[论文](https://arxiv.org/abs/1706.03762)  
![总体架构](./imgs/transformer_all_arch.png)

# 1、输入处理
## 1.1 词嵌入
> 将输入prompt向量化，$d_model$表示每个词向量的维度
## 1.2 位置嵌入
> 由于进行attention score计算时，每个q都可以与所有K进行交互，没有顺序概念，需要添加时间信息

### 1.2.1 绝对位置编码
> transformer使用正余弦位置编码获取位置信息，然后通过向量相加的方式将位置信息与词信息结合。这里是在**词嵌入向量中加入位置信息**
$$PE_{i, 2t} = \sin (k/10000^{2t/d})$$
$$PE_{i, 2t+1} = \cos (k/10000^{2t/d})$$
其中$PE_{i, 2t}$ 表示d维度向量 $PE_i$中第2t位置分量

In [ ]:
%pip install torch
import torch

def sinusodial_position_encoding(seq_len, d_model) -> torch.Tensor:
    encoding = torch.zeros(seq_len, d_model)
    pos = torch.arange(0, seq_len)
    pos = pos.float().unsqueeze(dim=1) # 第dim维添加一个维度
    print(f"pos={pos}\n")

    _2i = torch.arange(0, d_model, step=2).float()
    print(f"_2i={_2i}\n")

    encoding[:, 0::2] = torch.sin(pos / (10000 ** (_2i / d_model)))
    encoding[:, 1::2] = torch.cos(pos / (10000 ** (_2i / d_model)))

    return encoding

seq_len = 5
d_model = 4
pe = sinusodial_position_encoding(seq_len, d_model)
print(pe)


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Users/luocheng07/DuerE/.venv/lib/python3.9/site-packages/IPython/core/interactiveshell.py", line 3550, in run_code
  File "/var/folders/g0/7lqhg6q56s72yx4c3w8l5ng80000gn/T/ipykernel_4281/3987877745.py", line 1, in <module>
    import torch
ModuleNotFoundError: No module named 'torch'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/luocheng07/DuerE/.venv/lib/python3.9/site-packages/pygments/styles/__init__.py", line 45, in get_style_by_name
ModuleNotFoundError: No module named 'pygments.styles.default'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/luocheng07/DuerE/.venv/lib/python3.9/site-packages/IPython/core/interactiveshell.py", line 2144, in showtraceback
  File "/Users/luocheng07/DuerE/.venv/lib/python3.9/site-packages/IPython/core/ultratb.py", line 1435, in structured_traceback
  File "/Users/l

> 存在可训练的绝对位置BERT，暂时不表

### 1.2.2 相对位置编码
理论知识参考[RoPE](https://zhuanlan.zhihu.com/p/642884818)

In [ ]:
from math import cos
import torch

def rope(x: torch.Tensor) -> torch.Tensor:
    _, seq_len, d_model = x.size()
    thetas = (10000 ** (-torch.arange(0, d_model, 2).float() / d_model)).unsqueeze(0)
    thetas = torch.cat([thetas, thetas], dim=1)

    pos = torch.arange(1, seq_len + 1).float().squeeze(dim=1)
    thetas = thetas * pos

    cos_cached = torch.cos(thetas)
    sin_cached = torch.sin(thetas)

    x_cos = x * cos_cached
    x_sin = torch.cat([-x[:, :, d_model // 2 :], x[:, :, : d_model//2], -1]) * sin_cached

    o = x_cos + x_sin

    return o

